In [10]:
import numpy as np
import scipy.signal as sps
import scipy.stats as stats

#from tutorial https://kaavyamaha12.medium.com/extracting-audio-features-using-librosa-3be4ff1fe57f
def get_features(x, fs):
    try:
        feats = {}
        feats['len'] = len(x)
        feats['mean'] = np.mean(x)
        feats['std'] = np.std(x)
        feats['rms'] = np.sqrt(np.mean(x**2))
        feats['mav'] = np.mean(np.abs(x))
        feats['iEMG'] = np.sum(np.abs(x))
        feats['wl'] = np.sum(np.abs(np.diff(x)))
        feats['zcr'] = ((x[:-1]*x[1:]<0).sum()) 
        freqs, psd = sps.welch(x, fs=fs, nperseg=min(256, len(x)))
        total_power = np.sum(psd) + 1e-12
        feats['spec_entropy'] = -np.sum((psd/total_power) * np.log(psd/total_power + 1e-12))
        # median freq
        cumsum = np.cumsum(psd)
        median_freq = freqs[np.searchsorted(cumsum, total_power/2.0)]
        feats['median_freq'] = median_freq
        # skew/kurtosis
        feats['skew'] = stats.skew(x)
        feats['kurtosis'] = stats.kurtosis(x)
        return feats
    except Exception as e:
        print(f"Error during feature extraction {e}")
        return None
    

In [11]:
import os
import pandas as pd
import librosa

def check_labels(folder: str):
    f = "." 
    all_dfs = []
    normal_folder = os.path.join(f, "Normal")
    spont_folder = os.path.join(f, "Spontanaktivität")
    for file in os.listdir(folder):
        if file.endswith(".csv"):
            csv_path = os.path.join(folder, file)
            wav_file = os.path.splitext(file)[0] + ".wav"

            # check Normal folder
            normal_path = os.path.join(normal_folder, wav_file)
            spont_path = os.path.join(spont_folder, wav_file)

            if os.path.exists(normal_path):
                target_path = normal_path
            elif os.path.exists(spont_path):
                target_path = spont_path
            else:
                print(f"WAV file not found in either folder: {wav_file}")
                continue

            signal, fs= librosa.load(target_path)
            
            df = pd.read_csv(csv_path)
        
            features = []
            for index, row in df.iterrows():
                start_sample = int(row["Start Sample"])
                end_sample = int(row["End Sample"])
                segment = signal[start_sample:end_sample]
                segment_features = get_features(segment, fs)
                segment_features["file"] = target_path
                features.append(segment_features)

            features_df = pd.DataFrame(features)
            combined_df = pd.concat([df, features_df], axis=1)
            all_dfs.append(combined_df)

    final_df = pd.concat(all_dfs, ignore_index=True)
    final_df.to_csv("features.csv", index=False)
            
check_labels("./second_labeling")

In [12]:
import numpy as np
import pandas as pd
import librosa
import os

def add_random_negatives(csv_out="features.csv", total_negatives=1200):

    df = pd.read_csv(csv_out)
    all_neg_features = []

    # Group by file to process WAVs individually
    grouped = df.groupby("file")
    files = list(grouped.groups.keys())
    negatives_per_file = max(1, total_negatives // len(files))

    for file_path in files:
        # load WAV
        if not os.path.exists(file_path):
            print(f"WAV not found: {file_path}")
            continue
        signal, fs = librosa.load(file_path, sr=None)

        # get labeled intervals from the existing CSV
        file_rows = grouped.get_group(file_path)
        labeled_intervals = [(int(row["Start Sample"]), int(row["End Sample"])) for _, row in file_rows.iterrows()]
        lengths = [end - start for start, end in labeled_intervals]

        # generate random negatives
        features_neg = []
        attempts = 0
        while len(features_neg) < negatives_per_file and attempts < negatives_per_file * 5:
            attempts += 1
            seg_len = int(np.random.choice(lengths))
            max_start = len(signal) - seg_len
            if max_start <= 0:
                continue
            start = np.random.randint(0, max_start)
            end = start + seg_len

            # skip if overlapping labeled segments
            overlap = any(not (end <= s or start >= e) for s, e in labeled_intervals)
            if overlap:
                continue

            segment = signal[start:end]
            seg_features = get_features(segment, fs)
            seg_features["file"] = file_path
            seg_features["Label"] = "None"
            seg_features["Start Sample"] = start
            seg_features["End Sample"] = end
            seg_features["Start"] = start/fs
            seg_features["End"] = end/fs
            features_neg.append(seg_features)

        all_neg_features.extend(features_neg)

    # append negative features to the original CSV
    neg_df = pd.DataFrame(all_neg_features)
    combined_df = pd.concat([df, neg_df], ignore_index=True)
    combined_df.to_csv(csv_out, index=False)
    print(f"Saved combined CSV with negatives: {csv_out}")

add_random_negatives("features.csv")


Saved combined CSV with negatives: features.csv
